# 02 - Encoding into MySQL VECTOR(768) (Stage 2)

Start the store, encode both sides with the frozen native encoders, upsert
12,000 x 2 langs x 3 poolings = 72,000 vectors.

`pgvector` is PostgreSQL-only. MySQL 9 has a native `VECTOR` type plus
`STRING_TO_VECTOR`, `VECTOR_TO_STRING` and `DISTANCE`, which is what this uses.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config as C, data, db, mapping, pipeline
print(C.ROOT)

In [ ]:
import subprocess

out = subprocess.run(["docker", "compose", "up", "-d"], cwd=C.ROOT,
                     capture_output=True, text=True)
print(out.stdout[-800:])
print("server:", db.wait_ready())

In [ ]:
summary = pipeline.stage_encode()
summary

In [ ]:
pipeline.stage_verify(sample=50)

**Reading the output.** `paired_cosine_mean` is the cosine between the *unmapped*
English and Vietnamese vectors of the same sentence - the number the mapping has to
beat. `quantisation_error_max` should be ~0 (we wrote 6 decimals and got them back),
and `self_retrieval_top1` should be 1.0 (row alignment is intact).

Vectors are stored with 6 decimals, so ~1e-6 absolute error is baked into every
later number. MySQL 9.4 Community has `VECTOR` but no `DISTANCE()`, so scoring is
client-side exact cosine - no index, no approximation.
